# Step 6: Compare the models

Part of the model-selection steps in this folder (run in order the first time: step1 to step7). Shared setup (data,
random 80/20 train/test split by whole days, predictors, helpers) is in `model_utils.py`. The full explanation of
every step is in `docs/model_selection.md`.

**Needs:** Steps 2 and 5 (`step2_terms.json`, `stepwise.json`). **Saves:**
`data/processed/model_selection/chosen.json` (used by Step 7) and `docs/img/step6_model_comparison.png`.

| Measure | Data | Meaning | Better |
|---|---|---|---|
| Adjusted R² | train | share of demand explained, with a penalty for extra parameters | higher |
| AIC, BIC | train | fit with a penalty for extra parameters (BIC's penalty is larger) | lower |
| RMSE | test | typical prediction error in pickups per zone-hour; big misses count more | lower |
| MAE | test | average prediction error in pickups per zone-hour | lower |

The first three measure the fit on the data the model was trained on. RMSE and MAE measure predictions on days
the model hasn't seen, which is what matters for forecasting, so **test RMSE picks the model**.

**Baseline:** predict each zone-hour with the pickups from the same hour last week. A useful model must beat it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import model_utils as mu

train, test = mu.load()

In [ ]:
from statsmodels.tools.eval_measures import meanabs, rmse

full_terms = mu.read("step2_terms", made_by="step2_collinearity.ipynb")
stepwise = mu.read("stepwise", made_by="step5_stepwise.ipynb")
models = {f"S{i}: {p['change']}": p["terms"] for i, p in enumerate(stepwise["forward"], 1)}
models["Full model"] = full_terms
if stepwise["backward"]:
    models["Backward stepwise"] = stepwise["backward"][-1]["terms"]

names = {}  # one row per distinct model; duplicate names are joined with " = "
for name, terms in models.items():
    names.setdefault(frozenset(terms), []).append(name)

rows = []
for key, labels in names.items():
    res = mu.ols(models[labels[0]])
    pred = np.expm1(res.predict(test)).clip(lower=0)   # back from log(pickups + 1) to pickups
    rows.append({"model": " = ".join(labels), "parameters": len(res.params), "R2_adj": res.rsquared_adj,
                 "AIC": res.aic, "BIC": res.bic,
                 "RMSE_test": rmse(pred, test["pickups"]), "MAE_test": meanabs(pred, test["pickups"])})
    del res
rows.append({"model": "Baseline: same hour last week", "RMSE_test": rmse(test["lag168"], test["pickups"]),
             "MAE_test": meanabs(test["lag168"], test["pickups"])})
table = pd.DataFrame(rows).set_index("model")

best = table["RMSE_test"].drop("Baseline: same hour last week").idxmin()
mu.save("chosen", {"name": best, "terms": models[best.split(" = ")[0]]})
print(f"lowest test RMSE: {best} (saved for Step 7)")
table

In [ ]:
steps = table.loc[table.index.str.match(r"S\d")]
labels = steps.index.str.split(": ").str[1]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharex=True)
for ax, col, title in [(axes[0], "R2_adj", "Adjusted R² (train)"), (axes[1], "BIC", "BIC (train, lower is better)"),
                       (axes[2], "RMSE_test", "RMSE on test data (pickups)")]:
    ax.plot(range(len(steps)), steps[col], marker="o", ms=5, color=mu.BLUE)
    ax.set_title(title)
    ax.set_xticks(range(len(steps)), labels, rotation=60, ha="right", fontsize=8)
base_rmse = table.loc["Baseline: same hour last week", "RMSE_test"]
axes[2].axhline(base_rmse, color=mu.INK_2, lw=1.2, ls="--")
axes[2].annotate("last-week baseline", (0, base_rmse), xytext=(4, 4), textcoords="offset points",
                 color=mu.INK_2, fontsize=9)
fig.suptitle("Stepwise models: each point adds one term", x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout()
mu.save_figure(fig, "step6_model_comparison.png")
plt.show()

**Results.**
- **On the training data:** S12 (the full model without the two snow terms) has the lowest AIC and BIC, by a tiny
  margin over the full model. Adjusted R² is 0.916 for both.
- **On the test data, the full model has the lowest error:** RMSE **34.05** and MAE **19.20** pickups per zone-hour.
  S12 is practically the same (34.06), S2 (only the two lags) is 34.36, and the baseline is 40.12.
- **What does the work:** the two lags alone (S2) already cut the error from 40.1 (baseline) to 34.4. The calendar,
  zone and weather terms add a small further improvement. The differences between S9 and the full model are under
  0.1 pickups.

**Decision:** the **full model** is chosen. It has the lowest test error, and it is also the model used to explain
how each factor (rain, holidays, snow, ...) affects demand, so one model answers both questions.